# Test STAI with the Drive GGUF

This small Colab notebook loads the existing Q4_K_M GGUF from mounted Google Drive, asks you to upload `STAI_system_prompt.txt`, and uses that file as the system message for each test. The prompt is applied at inference time; it does not modify the GGUF weights.

Run the cells in order. This uses CPU inference so it can run without a GPU quota, though generation may be slower.


In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu llama-cpp-python
print('Updated CPU llama-cpp-python. If this kernel had llama_cpp imported already, restart the Colab runtime before continuing.')


In [ ]:
from google.colab import drive, files
from pathlib import Path
import re

drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
expected = MY_DRIVE / 'SVS-Cyber-Qwen3.5-2B-training' / 'gguf' / 'svs-cyber-v2-2B-Q4_K_M.gguf'
if expected.is_file():
    MODEL_PATH = expected
else:
    candidates = list(MY_DRIVE.rglob('*.gguf'))
    if not candidates:
        raise FileNotFoundError('No .gguf file found under MyDrive. Move or shortcut the model into My Drive, then rerun this cell.')
    print('GGUF files found:')
    for i, path in enumerate(candidates): print(f'[{i}] {path}')
    if len(candidates) == 1:
        MODEL_PATH = candidates[0]
    else:
        choice = int(input('Enter the number for the SVS-Cyber Q4_K_M model: '))
        MODEL_PATH = candidates[choice]
print('Selected GGUF:', MODEL_PATH, '| size:', f'{MODEL_PATH.stat().st_size / 1024**3:.2f} GiB')
expected_bytes = 1_274_396_352
actual_bytes = MODEL_PATH.stat().st_size
if MODEL_PATH.name == 'svs-cyber-v2-2B-Q4_K_M.gguf' and actual_bytes != expected_bytes:
    raise ValueError(f'GGUF size mismatch: got {actual_bytes:,} bytes; Drive copy is {expected_bytes:,} bytes. Re-copy/download the model, then retry.')
with MODEL_PATH.open('rb') as model_file:
    magic = model_file.read(4)
if magic != b'GGUF':
    raise ValueError(f'Not a GGUF file: header is {magic!r}. Check that MODEL_PATH points to the model file, not a Drive shortcut/HTML download.')
print('GGUF header and file size check passed.')

uploaded = files.upload()
prompt_files = [name for name in uploaded if name.lower().endswith('.txt')]
if not prompt_files:
    raise FileNotFoundError('Upload the generated STAI_system_prompt.txt file.')
PROMPT_PATH = Path(prompt_files[0])
SYSTEM_PROMPT = PROMPT_PATH.read_text(encoding='utf-8-sig').strip()
if not SYSTEM_PROMPT:
    raise ValueError('The system prompt file is empty.')
print('Loaded system prompt:', PROMPT_PATH.name, '| characters:', len(SYSTEM_PROMPT))


In [ ]:
import llama_cpp
from llama_cpp import Llama
print("llama-cpp-python version:", llama_cpp.__version__)

# CPU mode avoids requiring a Colab GPU. The GGUF already contains quantized weights.
try:
    llm = Llama(model_path=str(MODEL_PATH), n_ctx=4096, n_gpu_layers=0, verbose=True)
except Exception:
    print('Loader failed for:', MODEL_PATH)
    print('File size:', MODEL_PATH.stat().st_size, '| header:', MODEL_PATH.open('rb').read(4))
    raise

TEST_PROMPTS = [
    'Who are you, who built the STAI platform, and what is known about the model behind this file? Answer in English and separate confirmed facts from unknowns.',
    'A process alert is suspicious but there is not enough evidence to confirm compromise. Give a short defensive investigation plan. Answer in English.',
    'There is no scan output in the incident timeline. Can you say the scan passed? Answer in English.',
]

for prompt in TEST_PROMPTS:
    result = llm.create_chat_completion(
        messages=[{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':prompt}],
        temperature=0.2, max_tokens=220
    )
    answer = result['choices'][0]['message']['content'].strip()
    has_cjk = bool(re.search(r'[\u4e00-\u9fff]', answer))
    print('USER:', prompt, '\nSTAI:', answer)
    print('LANGUAGE CHECK:', 'FAIL — Chinese characters detected' if has_cjk else 'PASS — no Chinese characters detected')
    print('-' * 72)
